In [ ]:
# Import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
import warnings

warnings.filterwarnings('ignore')

# Matplotlib beállítások
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")
sns.set_palette("husl")

# Adatok betöltése
df = pd.read_csv('survey_data_cleaned.csv')

print("=" * 80)
print("ÁRKÉPZÉSI ANALÍZIS ÉS WILLINGNESS-TO-PAY (WTP) MODELLEZÉS")
print("=" * 80)

In [ ]:
# VAN WESTENDORP PRICE SENSITIVITY METER

print("\n" + "=" * 80)
print("1. VAN WESTENDORP PRICE SENSITIVITY METER")
print("=" * 80)

# Árpercepcióból kategóriák kialakítása
price_perception_mapping = {
    'Nem fizetnék érte semennyit': 'too_cheap',
    'Ennél sokkal kevesebbet fizetnék (max. 2–4 ezer Ft)': 'cheap',
    'Reális lenne egy kisebb ár (5–8 ezer Ft körül)': 'acceptable_low',
    '9-14 ezer Ft számomra megfelelő intervallum': 'acceptable_high',
    'Elfogadható a 15 ezer Ft feletti ár is': 'expensive'
}

# WTP eloszlás vizualizáció
wtp_sorted = np.sort(df['willingness_to_pay'].dropna())
cumulative = np.arange(1, len(wtp_sorted) + 1) / len(wtp_sorted) * 100
reverse_cumulative = 100 - cumulative

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Van Westendorp Price Sensitivity Meter', fontsize=16, fontweight='bold')

# 1.1 Kumulatív görbe
axes[0, 0].plot(wtp_sorted, cumulative, 'b-', linewidth=2, label='Túl olcsó (elfogadók %)')
axes[0, 0].plot(wtp_sorted, reverse_cumulative, 'r-', linewidth=2, label='Túl drága (elutasítók %)')
axes[0, 0].set_xlabel('Ár (Ft/hó)')
axes[0, 0].set_ylabel('Válaszadók aránya (%)')
axes[0, 0].set_title('Kumulatív WTP eloszlás', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].axhline(50, color='green', linestyle='--', alpha=0.5, label='50% küszöb')

# Optimális árpont (medián)
optimal_price = np.median(wtp_sorted)
axes[0, 0].axvline(optimal_price, color='green', linestyle='--', linewidth=2, 
                   label=f'Optimális: {optimal_price:.0f} Ft')
axes[0, 0].legend()

# 1.2 Price acceptance range
percentiles = [10, 25, 50, 75, 90]
price_points = [np.percentile(wtp_sorted, p) for p in percentiles]

axes[0, 1].barh(range(len(percentiles)), price_points, color='teal', alpha=0.7)
axes[0, 1].set_yticks(range(len(percentiles)))
axes[0, 1].set_yticklabels([f'{p}. percentilis' for p in percentiles])
axes[0, 1].set_xlabel('Ár (Ft/hó)')
axes[0, 1].set_title('Árelfogadási sávok (percentilisek)', fontweight='bold')
for i, (p, price) in enumerate(zip(percentiles, price_points)):
    axes[0, 1].text(price + 200, i, f'{price:.0f} Ft', va='center', fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# 1.3 Kereslet görbe (Demand curve)
price_range = np.linspace(0, df['willingness_to_pay'].max(), 100)
demand = [100 * (df['willingness_to_pay'] >= p).sum() / len(df) for p in price_range]

axes[1, 0].plot(price_range, demand, 'purple', linewidth=3)
axes[1, 0].set_xlabel('Ár (Ft/hó)')
axes[1, 0].set_ylabel('Potenciális vásárlók (%)')
axes[1, 0].set_title('Kereslet görbe', fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].fill_between(price_range, demand, alpha=0.3, color='purple')

# Kiemelt árpontok
for price in [3000, 5000, 8000, 10000]:
    pct = 100 * (df['willingness_to_pay'] >= price).sum() / len(df)
    axes[1, 0].plot(price, pct, 'ro', markersize=10)
    axes[1, 0].text(price, pct + 3, f'{price/1000:.0f}k Ft\n{pct:.1f}%', 
                    ha='center', fontsize=9, fontweight='bold')

# 1.4 Bevétel görbe (Revenue curve)
revenue = [p * (df['willingness_to_pay'] >= p).sum() for p in price_range]
max_revenue_idx = np.argmax(revenue)
optimal_revenue_price = price_range[max_revenue_idx]

axes[1, 1].plot(price_range, revenue, 'green', linewidth=3)
axes[1, 1].axvline(optimal_revenue_price, color='red', linestyle='--', linewidth=2,
                   label=f'Max bevétel: {optimal_revenue_price:.0f} Ft')
axes[1, 1].set_xlabel('Ár (Ft/hó)')
axes[1, 1].set_ylabel('Várható összbevétel (Ft)')
axes[1, 1].set_title('Bevételi görbe (Ár × Kereslet)', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)
axes[1, 1].fill_between(price_range, revenue, alpha=0.3, color='green')

plt.tight_layout()
plt.savefig('07_van_westendorp_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Optimális ár (medián alapján): {optimal_price:.0f} Ft/hó")
print(f"✓ Bevételoptimalizáló ár: {optimal_revenue_price:.0f} Ft/hó")
print(f"✓ Max várható bevétel: {revenue[max_revenue_idx]:,.0f} Ft")

In [ ]:
# REZERVÁCIÓS ÁR ÉS FOGYASZTÓI TÖBBLET

print("\n" + "=" * 80)
print("2. REZERVÁCIÓS ÁR ÉS FOGYASZTÓI TÖBBLET ANALÍZIS")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Rezervációs Ár és Fogyasztói Többlet', fontsize=16, fontweight='bold')

# 2.1 Rezervációs ár eloszlás (WTP = rezervációs ár)
axes[0, 0].hist(df['willingness_to_pay'].dropna(), bins=30, color='skyblue', 
                alpha=0.7, edgecolor='black', density=True)
axes[0, 0].axvline(df['willingness_to_pay'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Átlag: {df["willingness_to_pay"].mean():.0f} Ft')
axes[0, 0].axvline(df['willingness_to_pay'].median(), color='green', linestyle='--', 
                   linewidth=2, label=f'Medián: {df["willingness_to_pay"].median():.0f} Ft')
axes[0, 0].set_xlabel('Rezervációs ár (Ft/hó)')
axes[0, 0].set_ylabel('Sűrűség')
axes[0, 0].set_title('Rezervációs ár eloszlás (WTP)', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# 2.2 Fogyasztói többlet különböző árakon
test_prices = [2000, 3000, 5000, 7000, 10000, 15000]
consumer_surplus = []
market_size = []

for price in test_prices:
    buyers = df[df['willingness_to_pay'] >= price]
    surplus = (buyers['willingness_to_pay'] - price).sum()
    consumer_surplus.append(surplus)
    market_size.append(len(buyers))

axes[0, 1].bar(range(len(test_prices)), consumer_surplus, color='gold', alpha=0.8)
axes[0, 1].set_xticks(range(len(test_prices)))
axes[0, 1].set_xticklabels([f'{p/1000:.0f}k' for p in test_prices])
axes[0, 1].set_xlabel('Ár (Ft/hó)')
axes[0, 1].set_ylabel('Fogyasztói többlet (Ft)')
axes[0, 1].set_title('Fogyasztói többlet különböző árszinteken', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

for i, (surplus, size) in enumerate(zip(consumer_surplus, market_size)):
    axes[0, 1].text(i, surplus + 1000, f'{surplus:,.0f} Ft\n({size} fő)', 
                    ha='center', fontsize=9)

# 2.3 Piaci méret vs Ár
axes[1, 0].plot(test_prices, market_size, 'o-', color='coral', linewidth=2, markersize=10)
axes[1, 0].set_xlabel('Ár (Ft/hó)')
axes[1, 0].set_ylabel('Potenciális vásárlók száma')
axes[1, 0].set_title('Kereslet rugalmasság', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

for price, size in zip(test_prices, market_size):
    axes[1, 0].text(price, size + 0.3, f'{size}', ha='center', fontweight='bold')

# 2.4 Bevétel vs Profit (egyszerűsített)
# Feltételezzük: fix költség/fő = 500 Ft, változó költség = 20% a bevételből
fixed_cost_per_user = 500
variable_cost_rate = 0.20

revenues = [price * size for price, size in zip(test_prices, market_size)]
profits = [rev * (1 - variable_cost_rate) - (size * fixed_cost_per_user) 
           for rev, size in zip(revenues, market_size)]

x_pos = np.arange(len(test_prices))
width = 0.35

axes[1, 1].bar(x_pos - width/2, revenues, width, label='Bevétel', color='steelblue', alpha=0.8)
axes[1, 1].bar(x_pos + width/2, profits, width, label='Profit (becsült)', color='green', alpha=0.8)
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels([f'{p/1000:.0f}k' for p in test_prices])
axes[1, 1].set_xlabel('Ár (Ft/hó)')
axes[1, 1].set_ylabel('Összeg (Ft)')
axes[1, 1].set_title('Bevétel vs Profit (egyszerűsített modell)', fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)
axes[1, 1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('08_reservation_price_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Optimális profit keresése
max_profit_idx = np.argmax(profits)
optimal_profit_price = test_prices[max_profit_idx]

print(f"\n✓ Legnagyobb fogyasztói többlet: {max(consumer_surplus):,.0f} Ft ({test_prices[consumer_surplus.index(max(consumer_surplus))]} Ft áron)")
print(f"✓ Optimális profit ár: {optimal_profit_price} Ft/hó")
print(f"✓ Max profit: {profits[max_profit_idx]:,.0f} Ft")
print(f"✓ Piaci méret optimális áron: {market_size[max_profit_idx]} fő")

In [ ]:
# ÁRDISZKRIMINÁCIÓ ÉS SZEGMENTÁCIÓ

print("\n" + "=" * 80)
print("3. ÁRDISZKRIMINÁCIÓ ÉS TIERED PRICING STRATÉGIA")
print("=" * 80)

# Szegmensek szerinti WTP analízis
segments = {
    'Aktivitás': 'activity_level',
    'Tapasztalat': 'experience_level',
    'Közösség': 'community_preference'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Szegmens-specifikus Árképzési Stratégia', fontsize=16, fontweight='bold')

# 3.1 WTP aktivitás szerint
activity_wtp = df.groupby('activity_level')['willingness_to_pay'].agg(['mean', 'median', 'count', 'std'])
activity_wtp = activity_wtp.dropna()

x_pos = np.arange(len(activity_wtp))
axes[0, 0].bar(x_pos, activity_wtp['mean'], yerr=activity_wtp['std'], 
               capsize=5, color='steelblue', alpha=0.8, label='Átlag ± SD')
axes[0, 0].plot(x_pos, activity_wtp['median'], 'ro-', linewidth=2, 
                markersize=10, label='Medián')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(activity_wtp.index, rotation=45, ha='right')
axes[0, 0].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 0].set_title('WTP aktivitási szint szerint', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

for i, (mean, median, count) in enumerate(zip(activity_wtp['mean'], 
                                               activity_wtp['median'], 
                                               activity_wtp['count'])):
    axes[0, 0].text(i, mean + 500, f'{mean:.0f} Ft\n(n={count:.0f})', 
                    ha='center', fontsize=9)

# 3.2 WTP tapasztalat szerint
exp_wtp = df.groupby('experience_level')['willingness_to_pay'].agg(['mean', 'median', 'count', 'std'])
exp_wtp = exp_wtp.dropna()

x_pos = np.arange(len(exp_wtp))
axes[0, 1].bar(x_pos, exp_wtp['mean'], yerr=exp_wtp['std'], 
               capsize=5, color='coral', alpha=0.8, label='Átlag ± SD')
axes[0, 1].plot(x_pos, exp_wtp['median'], 'go-', linewidth=2, 
                markersize=10, label='Medián')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(exp_wtp.index, rotation=45, ha='right')
axes[0, 1].set_ylabel('Fizetési hajlandóság (Ft/hó)')
axes[0, 1].set_title('WTP tapasztalati szint szerint', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

for i, (mean, median, count) in enumerate(zip(exp_wtp['mean'], 
                                               exp_wtp['median'], 
                                               exp_wtp['count'])):
    axes[0, 1].text(i, mean + 500, f'{mean:.0f} Ft\n(n={count:.0f})', 
                    ha='center', fontsize=9)

# 3.3 Tiered Pricing javaslat
tiers = {
    'Basic': {
        'price': 2990,
        'target': 'Alkalmi fogadók',
        'features': ['Algoritmus tippek', 'Email support'],
        'color': 'lightblue'
    },
    'Standard': {
        'price': 5990,
        'target': 'Rendszeres fogadók',
        'features': ['Algoritmus + közösség', 'Bankroll management', 'Chat support'],
        'color': 'steelblue'
    },
    'Premium': {
        'price': 9990,
        'target': 'Aktív + veterán fogadók',
        'features': ['Mindent tartalmaz', 'Exkluzív tippek', 'Személyes tanácsadás', 'VIP közösség'],
        'color': 'darkblue'
    }
}

tier_names = list(tiers.keys())
tier_prices = [tiers[t]['price'] for t in tier_names]
tier_colors = [tiers[t]['color'] for t in tier_names]

# Becsült piaci méret minden tier-re
tier_market_size = [
    (df['willingness_to_pay'] >= tiers[tier]['price']).sum() 
    for tier in tier_names
]

axes[1, 0].bar(range(len(tier_names)), tier_prices, color=tier_colors, alpha=0.8)
axes[1, 0].set_xticks(range(len(tier_names)))
axes[1, 0].set_xticklabels(tier_names)
axes[1, 0].set_ylabel('Ár (Ft/hó)')
axes[1, 0].set_title('Ajánlott Tiered Pricing struktúra', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

for i, (price, target, size) in enumerate(zip(tier_prices, 
                                               [tiers[t]['target'] for t in tier_names],
                                               tier_market_size)):
    axes[1, 0].text(i, price + 300, f'{price} Ft\n{target}\n({size} fő)', 
                    ha='center', fontsize=8)

# 3.4 Bevételi potenciál tier-enként
# Feltételezés: konverziós ráta csökken magasabb áron
conversion_rates = [0.30, 0.20, 0.10]  # Basic, Standard, Premium
tier_revenues = [price * size * conv 
                 for price, size, conv in zip(tier_prices, tier_market_size, conversion_rates)]

axes[1, 1].bar(range(len(tier_names)), tier_revenues, color=tier_colors, alpha=0.8)
axes[1, 1].set_xticks(range(len(tier_names)))
axes[1, 1].set_xticklabels(tier_names)
axes[1, 1].set_ylabel('Havi bevétel (Ft)')
axes[1, 1].set_title('Becsült havi bevétel tier-enként\n(konverziós rátával)', fontweight='bold')
axes[1, 1].grid(axis='y', alpha=0.3)

for i, (rev, conv) in enumerate(zip(tier_revenues, conversion_rates)):
    axes[1, 1].text(i, rev + 500, f'{rev:,.0f} Ft\n({conv*100:.0f}% konv.)', 
                    ha='center', fontsize=9)

total_revenue = sum(tier_revenues)
axes[1, 1].axhline(total_revenue, color='green', linestyle='--', linewidth=2,
                   label=f'Összes: {total_revenue:,.0f} Ft')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('09_price_discrimination.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ AJÁNLOTT TIER STRUKTÚRA:")
for tier_name, tier_data in tiers.items():
    market = (df['willingness_to_pay'] >= tier_data['price']).sum()
    print(f"\n  {tier_name} - {tier_data['price']} Ft/hó")
    print(f"    Célcsoport: {tier_data['target']}")
    print(f"    Potenciális piac: {market} fő ({market/len(df)*100:.1f}%)")
    print(f"    Funkciók: {', '.join(tier_data['features'])}")

print(f"\n✓ Összes becsült havi bevétel (tiered): {total_revenue:,.0f} Ft")

In [ ]:
# PRICE ELASTICITY (ÁR RUGALMASSÁG)

print("\n" + "=" * 80)
print("4. ÁR RUGALMASSÁG ANALÍZIS")
print("=" * 80)

# Kereslet rugalmasság számítás
price_points = np.array([2000, 3000, 4000, 5000, 6000, 8000, 10000, 12000, 15000])
quantities = np.array([(df['willingness_to_pay'] >= p).sum() for p in price_points])

# Elasticitás számítás (point elasticity)
elasticities = []
for i in range(len(price_points) - 1):
    dQ = quantities[i+1] - quantities[i]
    dP = price_points[i+1] - price_points[i]
    P = price_points[i]
    Q = quantities[i]
    
    if Q != 0 and dQ != 0:
        elasticity = (dQ / Q) / (dP / P)
        elasticities.append(elasticity)
    else:
        elasticities.append(0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Ár Rugalmasság (Price Elasticity)', fontsize=16, fontweight='bold')

# 4.1 Kereslet görbe
axes[0].plot(price_points, quantities, 'o-', linewidth=2, markersize=10, color='purple')
axes[0].set_xlabel('Ár (Ft/hó)')
axes[0].set_ylabel('Kereslet (vásárlók száma)')
axes[0].set_title('Kereslet görbe', fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].fill_between(price_points, quantities, alpha=0.3, color='purple')

# 4.2 Rugalmassági együttható
axes[1].plot(price_points[:-1], elasticities, 's-', linewidth=2, markersize=10, color='red')
axes[1].axhline(-1, color='green', linestyle='--', linewidth=2, label='Egységnyi rugalmasság')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Ár (Ft/hó)')
axes[1].set_ylabel('Rugalmassági együttható (ε)')
axes[1].set_title('Price Elasticity of Demand', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

for price, elast in zip(price_points[:-1], elasticities):
    axes[1].text(price, elast - 0.2, f'{elast:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('10_price_elasticity.png', dpi=300, bbox_inches='tight')
plt.show()

avg_elasticity = np.mean([e for e in elasticities if e != 0])
print(f"\n✓ Átlagos ár rugalmasság: {avg_elasticity:.2f}")
if avg_elasticity < -1:
    print("  → Rugalmas kereslet: Árcsökkentés növeli a bevételt")
elif avg_elasticity > -1 and avg_elasticity < 0:
    print("  → Rugalmatlan kereslet: Áremelés növeli a bevételt")
else:
    print("  → Atipikus rugalmasság")

In [ ]:
# ÖSSZEFOGLALÓ AJÁNLÁSOK

print("\n" + "=" * 80)
print("5. ÁRKÉPZÉSI STRATÉGIAI AJÁNLÁSOK")
print("=" * 80)

print("\n📊 ALAPSTATISZTIKÁK:")
print(f"  • Átlagos WTP: {df['willingness_to_pay'].mean():.0f} Ft/hó")
print(f"  • Medián WTP: {df['willingness_to_pay'].median():.0f} Ft/hó")
print(f"  • Bevételoptimalizáló ár: {optimal_revenue_price:.0f} Ft/hó")
print(f"  • Profitoptimalizáló ár: {optimal_profit_price} Ft/hó")

print("\n💡 STRATÉGIAI AJÁNLÁSOK:")
print("\n  1. SINGLE PRICING opció:")
print(f"     → Javasolt ár: {int(np.median([optimal_price, optimal_revenue_price]))} Ft/hó")
print(f"     → Várható konverzió: {(df['willingness_to_pay'] >= np.median([optimal_price, optimal_revenue_price])).sum() / len(df) * 100:.1f}%")

print("\n  2. TIERED PRICING opció (AJÁNLOTT):")
print(f"     → Basic: {tiers['Basic']['price']} Ft/hó (konverzió: ~30%)")
print(f"     → Standard: {tiers['Standard']['price']} Ft/hó (konverzió: ~20%)")
print(f"     → Premium: {tiers['Premium']['price']} Ft/hó (konverzió: ~10%)")
print(f"     → Várható havi bevétel: {total_revenue:,.0f} Ft")

print("\n  3. LAUNCH STRATÉGIA:")
print(f"     → Early bird ár (első 3 hónap): {int(optimal_price * 0.7)} Ft/hó")
print(f"     → Profitgarancia: Első hónap pénzvisszafizetés")
print(f"     → Referral program: 1 hónap ingyen ajánlásért")

print("\n" + "=" * 80)
print("ÁRKÉPZÉSI ANALÍZIS BEFEJEZVE!")
print("Mentett képek:")
print("  - 07_van_westendorp_analysis.png")
print("  - 08_reservation_price_analysis.png")
print("  - 09_price_discrimination.png")
print("  - 10_price_elasticity.png")
print("=" * 80)